In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/CodeQwen1.5-7B-Chat",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/CodeQwen1.5-7B-Chat")



model.safetensors.index.json:   0%|          | 0.00/31.7k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.71G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

In [2]:
print(response)

Here's a quicksort algorithm implementation in Python:

```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)

print(quicksort([3,6,8,10,1,2,1]))
# Output: [1, 1, 2, 3, 6, 8, 10]
```

The algorithm works by selecting a "pivot" element from the array and partitioning the other elements into two sub-arrays, according to whether they are less than or greater than the pivot. The sub-arrays are then recursively sorted. This process continues until the sub-arrays become empty or contain just one element, which are then returned as they are already sorted.


In [ ]:
prompt = "Write a quicksort algorithm in python."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


In [10]:
def generate_sparql(question): 
    prompt = "generate a sparql query for the input question, please encolse the code in ``` and only show code not the detail:" + str(question)
    
    messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
        ]
    
    
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    if '```' in response:
        return response.split('```')[1]

    else:
        return response

In [11]:
generate_sparql('What is the capital city of Germany?')

'sparql\nPREFIX dbo: <http://dbpedia.org/ontology/>\nPREFIX dbr: <http://dbpedia.org/resource/>\n\nSELECT ?capital\nWHERE {\n  dbr:Germany dbo:capital ?capital .\n}\n'

In [12]:
import pandas as pd
import time

In [13]:
df_lcquad = pd.read_csv('Output/exp1_prompt_output_lcquad.csv')

In [14]:
len(df_lcquad)

5969

In [17]:
df_lcquad.columns

Index(['Unnamed: 0', 'questions', 'sparql_lcquad', 'output',
       'CodeQwen_sparql'],
      dtype='object')

In [ ]:
start_time = time.time()
df_lcquad['CodeQwen_sparql'] = df_lcquad["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

In [18]:
print(f"Execution time: {elapsed_time} seconds")

Execution time: 15905.595846414566 seconds


In [20]:
import nltk
from nltk.translate.bleu_score import sentence_bleu

#nltk.download('punkt')

for index, row in df_lcquad.iterrows():
    
    reference = nltk.word_tokenize(row['sparql_lcquad'].lower())
    if len(row['CodeQwen_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['CodeQwen_sparql'].lower())
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_lcquad.at[index, 'Org_bleu_score'] = bleu_score

/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of l

919
1241
1768
4600


In [21]:
average_bleu = df_lcquad['Org_bleu_score'].mean()
average_bleu

0.019446779454776988

In [22]:
df_lcquad.to_csv('Output/exp1_CodeQwen_lcquad.csv')

### Experiment 01 on CodeQwen code for dbpedia qald

In [23]:
df_qald = pd.read_csv('Output/exp1_prompt_output_qald.csv')

In [26]:
df_qald.columns

Index(['Unnamed: 0', 'id', 'questions', 'sparql_qald', 'sparql_gen', 'output',
       'CodeQwen_sparql'],
      dtype='object')

In [25]:
start_time = time.time()
df_qald['CodeQwen_sparql'] = df_qald["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

Execution time: 361.2628495693207 seconds


In [28]:
for index, row in df_qald.iterrows():
    
    reference = nltk.word_tokenize(row['sparql_qald'].lower())
    if len(row['CodeQwen_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['CodeQwen_sparql'].lower())
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_qald.at[index, 'Org_bleu_score'] = bleu_score

/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [29]:
average_bleu = df_qald['Org_bleu_score'].mean()
average_bleu

0.13010765704511987

In [36]:
df_qald.to_csv('Output/exp1_CodeQwen_qald.csv')

### Experiment 01 on misteral code for dbpedia Vquanda

In [30]:
df_vquanda = pd.read_csv('Output/exp1_prompt_output_vquanda.csv')

In [31]:
df_vquanda.head(2)


,Unnamed: 0,questions,sparql_vquanda,output
0,0,Count the number of people became famous for w...,SELECT DISTINCT COUNT(?uri) WHERE { ?x <http:/...,"[INST] \nYou are given an input question, conv..."
1,1,Which location city of Denver Broncos is the p...,SELECT DISTINCT ?uri WHERE { <http://dbpedia.o...,"[INST] \nYou are given an input question, conv..."


In [32]:
start_time = time.time()
df_vquanda['CodeQwen_sparql'] = df_vquanda["questions"].apply(generate_sparql)
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time} seconds")

Execution time: 2702.490475177765 seconds


In [33]:
for index, row in df_vquanda.iterrows():
    
    
    reference = nltk.word_tokenize(row['sparql_vquanda'].lower())
    
    if len(row['CodeQwen_sparql']) < 1:
        print(index)
        continue
    candidate = nltk.word_tokenize(row['CodeQwen_sparql'].lower())
    
    bleu_score = sentence_bleu([reference], candidate[1:])
    df_vquanda.at[index, 'Org_bleu_score'] = bleu_score
    

/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/upb/users/m/manzoor/profiles/unix/cs/.local/lib/python3.11/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [34]:
average_bleu = df_vquanda['Org_bleu_score'].mean()
average_bleu

0.032080089902087223

In [35]:
df_vquanda.to_csv('Output/exp1_CodeQwen_vquanda.csv')